In [2]:
from pyspark.sql import SparkSession
import os

# Nạp đầy đủ Package cho Kafka, Iceberg, Nessie và S3 (MinIO)
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1,'
    'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,'
    'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.67.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4 '  # THƯ VIỆN CÒN THIẾU Ở ĐÂY
    'pyspark-shell'
)

spark = SparkSession.builder \
    .appName("Fix-S3A-Error") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://silver/") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Spark Session đã khởi tạo lại với thư viện S3A!")

✅ Spark Session đã khởi tạo lại với thư viện S3A!


In [6]:
spark.sql("SHOW TABLES IN nessie").show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|bronze_db| amazon_purchase_raw|      false|
|  gold_db|        dim_customer|      false|
|  gold_db|        dim_location|      false|
|  gold_db|         dim_product|      false|
|  gold_db|            dim_time|      false|
|  gold_db|          fact_order|      false|
|silver_db|amazon_purchase_s...|      false|
|silver_db|       survey_silver|      false|
+---------+--------------------+-----------+



In [7]:
# 1. Xóa các bảng trong gold_db (Đúng Catalog và Namespace)
#spark.sql("DROP TABLE IF EXISTS nessie.gold_db.dim_customer")
#spark.sql("DROP TABLE IF EXISTS nessie.gold_db.dim_location")
#spark.sql("DROP TABLE IF EXISTS nessie.gold_db.dim_product")
#spark.sql("DROP TABLE IF EXISTS nessie.gold_db.dim_time")
# Thử dùng DROP TABLE nhưng không quan tâm đến metadata cũ
spark.sql("DROP TABLE IF EXISTS nessie.silver_db.amazon_purchase_silver")

print("✅ Đã xóa sạch các bảng lớp Gold trên Nessie!")

# 2. Lưu ý về Checkpoint (Quan trọng)
# Để Spark nạp lại từ Silver sang Gold từ đầu, Thăng nên đổi tên 
# checkpointLocation trong code Gold thành một version mới (v6, v7...)
# Hoặc xóa folder này trong MinIO: s3a://gold/checkpoints/silver_to_gold_final_v5

✅ Đã xóa sạch các bảng lớp Gold trên Nessie!


In [8]:
spark.sql("SHOW TABLES IN nessie").show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|bronze_db|amazon_purchase_raw|      false|
|  gold_db|       dim_customer|      false|
|  gold_db|       dim_location|      false|
|  gold_db|        dim_product|      false|
|  gold_db|           dim_time|      false|
|  gold_db|         fact_order|      false|
|silver_db|      survey_silver|      false|
+---------+-------------------+-----------+

